### Imports

In [1]:
from pathlib import Path
import pandas as pd

### Daten einlesen

In [2]:
df_categories = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/categories.csv")
df_customers = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/customers.csv", dtype={"customer_id": "string"})
df_geolocation = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/geolocation.csv")
df_items = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/items.csv")
df_orders = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/orders.csv", dtype={"order_id": "string", "customer_id": "string"})
df_payments = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/payments.csv")
df_pilot_support = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/pilot_support.csv")
df_products = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/products.csv")
df_reviews = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/reviews.csv")
df_sellers = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/sellers.csv")
df_service_experiment = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/service_experiment.csv")
df_service_observations = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/service_observation.csv")

## 2.1 Daten vorbereiten

### a) Datenqualität prüfen und bereinigen

1. Was erwarte ich, bzw. Was sagt eine Zeile aus?
2. Woran erkenne ich eine Zeile eindeutig?
3. Wo weichen meine Daten ab?
4. Was kann ich daruas begründet entscheiden?

In [16]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
# 1. Eine Zeile soll eine Bestellung darstellen.
print("Bestellungen insgesamt:", len(df_orders))

# 2. Ist die Bestellnummer vollständig und eindeutig?
ids = df_orders["order_id"]
print("Fehlende Bestellnummern:", ids.isna().sum())
print("Mehrfach vorkommende Bestellnummern:", ids[ids.duplicated()].nunique())

# 3. Passen Status und tatsächliches Lieferdatum zusammen?
geliefert = df_orders["order_status"].eq("delivered")
storniert = df_orders["order_status"].eq("canceled")
ist_vorhanden = df_orders["order_delivered_customer_date"].notna()
soll_vorhanden = df_orders["order_estimated_delivery_date"].notna()

print("Geliefert, aber ohne Lieferdatum:", (geliefert & ~ist_vorhanden).sum())
print("Storniert, aber mit Lieferdatum:", (storniert & ist_vorhanden).sum())

# 4. Welche Bestellungen eignen sich für den Vergleich von Ist- und Solltermin?
fuer_liefervergleich = geliefert & ist_vorhanden & soll_vorhanden   # orders nicht "bereinigt", nur Zeilen mit vollständigen und konsistenten Daten zwischengespeichert
print("Geliefert und beide Termine vorhanden:", fuer_liefervergleich.sum())

Bestellungen insgesamt: 99441
Fehlende Bestellnummern: 0
Mehrfach vorkommende Bestellnummern: 0
Geliefert, aber ohne Lieferdatum: 8
Storniert, aber mit Lieferdatum: 6
Geliefert und beide Termine vorhanden: 96470


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine Bestellung. Bei einer zugestellten Bestellung erwarte ich ein tatsächliches Lieferdatum.
2. **Was macht die Zeile eindeutig?** Die `order_id` ist bei allen 99.441 Bestellungen vorhanden und eindeutig.
3. **Wo weichen die Daten ab?** Bei 8 als zugestellt markierten Bestellungen fehlt das Lieferdatum. Bei 6 stornierten Bestellungen ist ein Lieferdatum eingetragen.
4. **Was entscheide ich?** Für den Vergleich von zugesagtem und tatsächlichem Liefertermin verwende ich die 96.470 zugestellten Bestellungen, bei denen beide Termine vorliegen. Für eine Korrektur der auffälligen Einträge fehlt ein Beleg.

In [18]:
df_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ce02b2c6e70d5b2fe24d103952d389bf,1,70d17ed46123e5b68dff23e113415d5a,3d871de0142ce09b7081e2b9d1733cb1,2017-08-03 13:45:11,54.90,12.72
1,4886c605fd1f7edd568a936814a015b7,1,bb09b0b5f3668effd704b1b2a83cae9b,6560211a19b47992c3666cc44a7e94c0,2017-12-22 02:31:10,189.00,8.58
2,2abac84637c5e0ddd2923f22ba61a96a,1,4ae634441e444ca4bc85903cafe98d73,92eb0f42c21942b6552362b9b114707d,2017-04-05 01:22:35,25.00,16.05
3,906dfb858a6afcaea1fb5cc0ed730147,1,33e77eecc23c25a65d10f627e5053617,7c67e1448b00f6e969d365cea6b010ab,2017-11-01 02:49:41,114.94,28.16
4,dcc495566353215aa81cab568f8fb2bf,1,6fd08d44046ab994b96ff38ad6fcfba1,d9a84e1403de8da0c3aa531d6d108ba6,2018-03-30 16:08:15,39.90,18.23


In [19]:
schluessel = ["order_id", "order_item_id"]

print("Positionszeilen insgesamt:", len(df_items))
print("Zeilen mit unvollständigem Schlüssel:", df_items[schluessel].isna().any(axis=1).sum())
print("Vollständig identische Wiederholungen:", df_items.duplicated().sum())

# Nur vollständig identische Zeilen entfernen
df_items_clean = df_items.drop_duplicates().copy()

print("Positionen nach Bereinigung:", len(df_items_clean))
print("Verbleibende Schlüsselwiederholungen:", df_items_clean.duplicated(schluessel).sum())

# Auswirkung auf den gesamten Warenwert zeigen
print("Warenwert vorher (BRL):", round(df_items["price"].sum(), 2))
print("Warenwert nachher (BRL):", round(df_items_clean["price"].sum(), 2))

Positionszeilen insgesamt: 112875
Zeilen mit unvollständigem Schlüssel: 0
Vollständig identische Wiederholungen: 225
Positionen nach Bereinigung: 112650
Verbleibende Schlüsselwiederholungen: 0
Warenwert vorher (BRL): 13619897.04
Warenwert nachher (BRL): 13591643.7


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine Position innerhalb einer Bestellung mit Produkt, Händler, Preis und Frachtkosten.
2. **Was macht die Zeile eindeutig?** Die Kombination aus Bestellnummer (`order_id`) und Positionsnummer (`order_item_id`).
3. **Wo weichen die Daten ab?** Unter den 112.875 Positionszeilen gibt es 225 vollständig identische Wiederholungen. Beim Summieren würden ihre Preise und Frachtkosten doppelt zählen.
4. **Was entscheide ich?** Ich entferne die identischen Wiederholungen. Es bleiben 112.650 Positionen mit eindeutigem Schlüssel. Für die weitere Analyse verwende ich `df_items_clean`.

In [20]:
df_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [21]:
ids = df_customers["customer_id"]

print("Kundeneinträge:", len(df_customers))
print("Fehlende customer_id:", ids.isna().sum())
print("Wiederholte customer_id:", ids.dropna().duplicated().sum())
print("Fehlender Bundesstaat:", df_customers["customer_state"].isna().sum())
print("Bestellungen ohne passenden Kundeneintrag:",
      (~df_orders["customer_id"].isin(ids.dropna())).sum())

Kundeneinträge: 99441
Fehlende customer_id: 0
Wiederholte customer_id: 0
Fehlender Bundesstaat: 0
Bestellungen ohne passenden Kundeneintrag: 0


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt den Kundeneintrag zu einer Bestellung. Dieselbe Person kann bei mehreren Bestellungen vorkommen.
2. **Was macht die Zeile eindeutig?** Die `customer_id`. Die `customer_unique_id` erkennt dieselbe Person über mehrere Bestellungen hinweg und darf sich wiederholen.
3. **Wo weichen die Daten ab?** Bei den geprüften Feldern gibt es keine Auffälligkeiten. Jede `customer_id` ist vorhanden und eindeutig, jeder Eintrag enthält einen Bundesstaat und jede Bestellung hat einen passenden Kundeneintrag.
4. **Was entscheide ich?** Für diese Felder ist keine Bereinigung nötig. Ich verwende die `customer_id`, um jeder Bestellung den Bundesstaat des Kunden zuzuordnen.

In [22]:
df_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [23]:
ids = df_sellers["seller_id"]

print("Händlereinträge:", len(df_sellers))
print("Fehlende seller_id:", ids.isna().sum())
print("Wiederholte seller_id:", ids.dropna().duplicated().sum())
print("Fehlender Bundesstaat:", df_sellers["seller_state"].isna().sum())
print("Positionen ohne passenden Händler:",
      (~df_items_clean["seller_id"].isin(ids.dropna())).sum())

Händlereinträge: 3095
Fehlende seller_id: 0
Wiederholte seller_id: 0
Fehlender Bundesstaat: 0
Positionen ohne passenden Händler: 0


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt einen Händler und seinen Standort.
2. **Was macht die Zeile eindeutig?** Die `seller_id` identifiziert jeden Händler eindeutig.
3. **Wo weichen die Daten ab?** Alle Händler-IDs sind vorhanden und eindeutig. Jeder Händler hat einen eingetragenen Bundesstaat, und jede Bestellposition lässt sich einem Händler zuordnen.
4. **Was entscheide ich?** Für diese Felder ist keine Bereinigung nötig. Ich nutze `seller_id`, um den Händlerstandort jeder Position mit dem Bundesstaat des Kunden zu vergleichen.

In [24]:
df_reviews.head()

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53


In [25]:
anzahl = df_reviews.groupby("order_id").size()

print("Bewertungszeilen:", len(df_reviews))
print("Zeilen ohne Bestellnummer:", df_reviews["order_id"].isna().sum())
print("Bestellungen mit mehreren Bewertungen:", (anzahl > 1).sum())
print("Fehlende Abgabezeitpunkte:", df_reviews["review_answer_timestamp"].isna().sum())
print("Fehlende oder ungültige Sternezahl:",
      (~df_reviews["review_score"].isin([1, 2, 3, 4, 5])).sum())

Bewertungszeilen: 99224
Zeilen ohne Bestellnummer: 0
Bestellungen mit mehreren Bewertungen: 547
Fehlende Abgabezeitpunkte: 0
Fehlende oder ungültige Sternezahl: 0


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine abgegebene Bewertung zu einer Bestellung.
2. **Was macht die Zeile eindeutig?** Einen eindeutigen Einzelschlüssel gibt die Aufgabenstellung für diese Tabelle nicht vor. Die `order_id` darf mehrfach vorkommen, und die `review_id` ist nicht global eindeutig.
3. **Wo weichen die Daten ab?** Bei 547 Bestellungen liegen mehrere Bewertungen vor. Das ist zulässig. Bei allen 99.224 Bewertungszeilen sind Bestellnummer und Abgabezeitpunkt vorhanden.
4. **Was entscheide ich?** In 2.1b wähle ich je Bestellung die Bewertung mit dem spätesten Abgabezeitpunkt. Bei Gleichstand nehme ich die größere `review_id`. Anschließend prüfe ich, ob diese Bewertung 1 bis 5 Sterne hat.

In [26]:
df_products.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0,1.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0,1.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0,1.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0,1.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0,4.0


In [ ]:
ids = df_products["product_id"]
kategorie = df_products["product_category_name"]
referenz = df_categories["product_category_name"].dropna()

unbekannt = kategorie.notna() & ~kategorie.isin(referenz)   # stimmt nicht mit df_categories überein
normalisiert = kategorie.str.strip().str.lower()
korrigierbar = unbekannt & normalisiert.isin(referenz)

print("Produkte:", len(df_products))
print("Fehlende Produkt-IDs:", ids.isna().sum())
print("Wiederholte Produkt-IDs:", ids.dropna().duplicated().sum())
print("Fehlende Kategorien:", kategorie.isna().sum())
print("Unbekannte Kategorienamen (Anzahl Produkte):", unbekannt.sum())
print("Davon eindeutig korrigierbar:", korrigierbar.sum())

Produkte: 32951
Fehlende Produkt-IDs: 0
Wiederholte Produkt-IDs: 0
Fehlende Kategorien: 610
Unbekannte Kategorienamen (Anzahl Produkte): 330
Davon eindeutig korrigierbar: 220


In [ ]:
# Bereinigung der Produktkategorien in df_products anhand der Referenzliste df_categories
df_products_clean = df_products.copy()
df_products_clean.loc[korrigierbar, "product_category_name"] = normalisiert[korrigierbar]

# Überprüfung der unbekannten Kategorien nach der Bereinigung
danach = df_products_clean["product_category_name"]
print("Unbekannte Kategorienamen nach Bereinigung:",
      (danach.notna() & ~danach.isin(referenz)).sum())

Unbekannte Kategorienamen nach Bereinigung: 110


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt ein Produkt mit Kategorie und weiteren Merkmalen wie Gewicht und Maßen.
2. **Was macht die Zeile eindeutig?** Die `product_id` identifiziert das Produkt.
3. **Wo weichen die Daten ab?** Bei 330 Produkten stimmt der eingetragene Kategoriename mit keinem Namen in der Referenztabelle überein.
4. **Was entscheide ich?** Bei 220 Produkten ergibt sich durch Kleinschreibung und Entfernen äußerer Leerzeichen ein exakter Treffer. Ich korrigiere diese Schreibweisen. Damit sinkt die Zahl der Produkte mit unbekanntem Kategorienamen von 330 auf 110.

In [30]:
df_categories.head()

,product_category_name,product_category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


In [31]:
namen = df_categories["product_category_name"]
uebersetzung_fehlt = df_categories["product_category_name_english"].isna()

print("Kategorien insgesamt:", len(df_categories))
print("Fehlende Kategorienamen:", namen.isna().sum())
print("Wiederholte Kategorienamen:", namen.dropna().duplicated().sum())
print("Fehlende Übersetzungen:", uebersetzung_fehlt.sum())
print("Betroffene Kategorien:", namen[uebersetzung_fehlt].tolist())

Kategorien insgesamt: 73
Fehlende Kategorienamen: 0
Wiederholte Kategorienamen: 0
Fehlende Übersetzungen: 2
Betroffene Kategorien: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


1. **Was sagt eine Zeile aus?** Eine Zeile ordnet einer portugiesischen Produktkategorie ihre englische Übersetzung zu.
2. **Was macht die Zeile eindeutig?** Der portugiesische Kategoriename `product_category_name`.
3. **Wo weichen die Daten ab?** Alle 73 Kategorienamen sind vorhanden und eindeutig. Bei zwei Kategorien fehlt die englische Übersetzung.
4. **Was entscheide ich?** Ich behalte beide Einträge. Für den Abgleich mit `products` verwende ich die portugiesischen Namen, die auch bei diesen beiden Kategorien vorliegen.

### b) Eine Tabelle mit je einer Zeile pro Bestellung

In [ ]:
# Aggregation der Bestellpositionen je Bestellung
positionen_je_bestellung = (
    df_items_clean.groupby("order_id", as_index=False)
    .agg(
        positionszahl=("order_item_id", "size"),
        warenwert=("price", "sum"),
        frachtkosten=("freight_value", "sum")
    )
)

print("Eine Zeile je Bestellung:", positionen_je_bestellung["order_id"].is_unique)
print("Alle Positionen erfasst:",
      positionen_je_bestellung["positionszahl"].sum() == len(df_items_clean))

positionen_je_bestellung.head()

Eine Zeile je Bestellung: True
Alle Positionen erfasst: True


,order_id,positionszahl,warenwert,frachtkosten
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [ ]:
# Zusammenführung der Bestellungen mit den aggregierten Bestellpositionen
bestelltabelle = df_orders.merge(
    positionen_je_bestellung,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Bestellungen vorher:", len(df_orders))
print("Bestellungen nachher:", len(bestelltabelle))
print("Bestellungen ohne Positionen:", bestelltabelle["positionszahl"].isna().sum())

bestelltabelle.head()

Bestellungen vorher: 99441
Bestellungen nachher: 99441
Bestellungen ohne Positionen: 775


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,positionszahl,warenwert,frachtkosten
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,19.90,8.72


In [ ]:
ohne_position = bestelltabelle["positionszahl"].isna()  # Bestellungen ohne Positionen
bestelltabelle.loc[ohne_position, "order_status"].value_counts(dropna=False)  # Zählt die Bestellstatus der Bestellungen ohne Positionen

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [ ]:
# Zusammenführung der Bestellungen mit den Kundeninformationen (Bundesstaat)
bestellungen_mit_kunden = bestelltabelle.merge(
    df_customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Bestellungen:", len(bestellungen_mit_kunden))
print("Ohne Kundenbundesstaat:", bestellungen_mit_kunden["customer_state"].isna().sum())

Bestellungen: 99441
Ohne Kundenbundesstaat: 0


In [ ]:
# Zusammenführung der Bestellpositionen mit den Händlerinformationen (Bundesstaat)
positionen_mit_haendlern = df_items_clean.merge(
    df_sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("Positionen:", len(positionen_mit_haendlern))
print("Ohne Händlerbundesstaat:", positionen_mit_haendlern["seller_state"].isna().sum())

Positionen: 112650
Ohne Händlerbundesstaat: 0


In [ ]:
# Zusammenführung der Bestellpositionen mit den Kunden- und Händlerinformationen (Bundesstaat)

positionen = positionen_mit_haendlern.merge(
    bestellungen_mit_kunden[["order_id", "customer_state"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

assert positionen["customer_state"].notna().all()
positionen["anderer_bundesstaat"] = positionen["seller_state"] != positionen["customer_state"]

# any(): Eine abweichende Position genügt für die gesamte Bestellung.
versand_je_bestellung = positionen.groupby("order_id")["anderer_bundesstaat"].any()

bestellungen_mit_versand = bestellungen_mit_kunden.copy()
bestellungen_mit_versand["versand_zwischen_bundesstaaten"] = (
    bestellungen_mit_versand["order_id"].map(versand_je_bestellung).astype("boolean")
)

print(bestellungen_mit_versand["versand_zwischen_bundesstaaten"].value_counts(dropna=False))

versand_zwischen_bundesstaaten
True     63313
False    35353
<NA>       775
Name: count, dtype: Int64


In [ ]:
# Zusammenführung der Bestellungen mit den Bewertungen (Sternezahl)

reviews = df_reviews.copy()
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

letzte_bewertung = (
    reviews.sort_values(["order_id", "review_answer_timestamp", "review_id"])
    .drop_duplicates("order_id", keep="last")
)

bestellungen_mit_bewertung = bestellungen_mit_versand.merge(
    letzte_bewertung[["order_id", "review_score"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Bestellungen:", len(bestellungen_mit_bewertung))
print("Ohne Sternezahl:", bestellungen_mit_bewertung["review_score"].isna().sum())

Bestellungen: 99441
Ohne Sternezahl: 768


In [ ]:
# Auswahl der Bestellungen für die Analyse basierend auf Lieferung und Bewertung

basis = bestellungen_mit_bewertung

lieferung_ok = (
    basis["order_status"].eq("delivered")
    & basis["order_delivered_customer_date"].notna()
    & basis["order_estimated_delivery_date"].notna()
)
bewertung_ok = basis["review_score"].isin([1, 2, 3, 4, 5])

analyse_bestellungen = basis.loc[lieferung_ok & bewertung_ok].copy()

print("Geliefert mit beiden Terminen:", lieferung_ok.sum())
print("Davon ohne gültige letzte Bewertung:", (lieferung_ok & ~bewertung_ok).sum())
print("Bestellungen für die Analyse:", len(analyse_bestellungen))

Geliefert mit beiden Terminen: 96470
Davon ohne gültige letzte Bewertung: 646
Bestellungen für die Analyse: 95824


In [45]:
#analyse_bestellungen.describe()
analyse_bestellungen.info()
#analyse_bestellungen.head()

<class 'pandas.DataFrame'>
Index: 95824 entries, 0 to 99440
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   order_id                        95824 non-null  object 
 1   customer_id                     95824 non-null  string 
 2   order_status                    95824 non-null  str    
 3   order_purchase_timestamp        95824 non-null  str    
 4   order_approved_at               95810 non-null  str    
 5   order_delivered_carrier_date    95823 non-null  str    
 6   order_delivered_customer_date   95824 non-null  str    
 7   order_estimated_delivery_date   95824 non-null  str    
 8   positionszahl                   95824 non-null  float64
 9   warenwert                       95824 non-null  float64
 10  frachtkosten                    95824 non-null  float64
 11  customer_state                  95824 non-null  str    
 12  versand_zwischen_bundesstaaten  95824 non-null  

### Zusammenfassung 2.1 b) Bestelltabelle erstellen

Wir haben die bereinigten Positionen je Bestellung zu Positionszahl, Warenwert und Frachtkosten zusammengefasst. Mithilfe der Kunden- und Händlerstandorte haben wir die Versandgruppe bestimmt und jeder Bestellung, soweit vorhanden, die zuletzt abgegebene Bewertung zugeordnet. Die Bestelltabelle enthält alle **99.441 Bestellungen mit genau einer Zeile je Bestellung**.

Von den **96.470** gelieferten Bestellungen mit tatsächlichem und zugesagtem Liefertermin hatten **646** keine gültige letzte Bewertung. Für die Analyse bleiben **95.824 Bestellungen**.